# 빈도·밀도 분석

- 대응 코드: `frequency.py`
- 문서·선택된 키워드 언급 수를 확인하고 빈도표와 밀도표를 점검.
- 전체 결과 재생성은 선택 실행.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

# 작업 폴더가 저장소 루트이거나 노트북 폴더여도 analysis/를 찾는다.
candidates = [Path.cwd(), *Path.cwd().parents]
analysis_root = next((candidate for base in candidates
                      for candidate in (base, base / "code" / "analysis", base / "ESG_comp" / "code" / "analysis")
                      if (candidate / "common" / "config.py").is_file()), None)
if analysis_root is None:
    raise FileNotFoundError("ESG_comp/code/analysis 폴더에서 노트북을 열어 주세요.")
if not os.environ.get("ESG_RESEARCH_ROOT"):
    raise RuntimeError("비공개 입력 위치를 ESG_RESEARCH_ROOT 환경변수로 먼저 지정하세요.")
sys.path.insert(0, str(analysis_root))
from common import config as cfg
from common.io_common import load_enriched


def show_csv(path, rows=10, columns=None):
    path = Path(path)
    if not path.is_file():
        print(f"저장 결과 없음: {path}\n입력 자료와 앞 단계를 확인한 뒤 실행하세요.")
        return None
    frame = pd.read_csv(path)
    display(frame[columns].head(rows) if columns else frame.head(rows))
    print(f"{path.name}: {len(frame):,}행")
    return frame


In [ ]:
LANG = "kr"  # "kr" 또는 "jp"
RUN_REPORT = False
PREVIEW_ROWS = 10

In [ ]:
from stage2_frequency import frequency as step
if LANG not in cfg.LANGS:
    raise ValueError(f"LANG은 {cfg.LANGS} 중 하나여야 합니다.")
merged, docs = load_enriched(LANG)
print(f"문서 {len(docs):,}건 · 선택된 언급 {len(merged):,}건")
display(merged.groupby("category").size().sort_values(ascending=False).head(PREVIEW_ROWS).rename("mentions").to_frame())
display(docs[[c for c in ("doc_id", "target_field_primary", "page_type_reviewed", "full_text_len") if c in docs]].head(PREVIEW_ROWS))

In [ ]:
if RUN_REPORT:
    results = step.build_stage2_report(LANG)
    display(results["category"].head(PREVIEW_ROWS))
    display(results["target_field_density"].head(PREVIEW_ROWS))
else:
    print("전체 빈도·밀도 결과 재생성은 RUN_REPORT=True로 선택하세요.")

In [ ]:
out_dir = cfg.OUTPUT_DIR / LANG / "단계2_빈도분석"
for filename in (f"1_ESG범주별_개요_{LANG}.csv", f"2_학문분야별_밀도_{LANG}.csv", f"5_페이지타입별_개요_{LANG}.csv"):
    show_csv(out_dir / filename, PREVIEW_ROWS)